In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm import tqdm

# -----------------------------
# 1) Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# -----------------------------
# 2) Dataset / Transform
#    - MNIST (28x28) -> 32x32 (padding 2px each side)
#    - ToTensor() => float32 [0,1]
# -----------------------------
BATCH_SIZE = 128
NUM_WORKERS = 0  # 자동 (병렬처리를 수동으로 하려면 2~8)

transform = transforms.Compose([
    transforms.Pad(2),          # 28x28 -> 32x32 (resize_with_pad와 동일 목적)
    transforms.ToTensor(),      # (H,W) -> (1,H,W), float32, [0,1 정규화]
])

full_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set   = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# TF의 validation_split=0.1과 동일하게 90/10 분리
train_size = int(len(full_train) * 0.9)
val_size   = len(full_train) - train_size
train_set, val_set = random_split(full_train, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=False)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=False)

# -----------------------------
# 3) LeNet
#    Conv(6,5) -> AvgPool(2)
#    Conv(16,5) -> AvgPool(2)
#    Conv(120,5)
#    Flatten -> FC(84) -> FC(10)
#    Activation: tanh
# -----------------------------
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # 합성곱 층 1
            nn.Conv2d(1, 6, kernel_size=5),   # 32->28
            nn.Tanh(),
            nn.AvgPool2d(2),                  # 28->14

            # 합성곱 층 2
            nn.Conv2d(6, 16, kernel_size=5),  # 14->10
            nn.Tanh(),
            nn.AvgPool2d(2),                  # 10->5

            # 완전연결층 1
            nn.Conv2d(16, 120, kernel_size=5),# 5->1
            nn.Tanh(),
        )

        # 분류기 (완전연결층2 & 출력층)
        self.classifier = nn.Sequential(
            nn.Flatten(),                     # 120*1*1 -> 120
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = LeNet().to(device)
print(model)

# -----------------------------
# 4) Loss / Optimizer
#    - TF sparse_categorical_crossentropy == PyTorch CrossEntropyLoss
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -----------------------------
# 5) Train / Eval loops
# -----------------------------
def run_epoch(model, loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)

            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)  # 손실 누적 (샘플 가중)
        pred = logits.argmax(dim=1)  # 모델이 예측한 라벨 인덱스
        correct += (pred == y).sum().item()  # 정답 수
        total += x.size(0)  # 배치 수

    return total_loss / total, correct / total  # 평균 손실 / 정확도

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, train=True)
    va_loss, va_acc = run_epoch(model, val_loader, train=False)
    print(f"[{epoch}/{EPOCHS}] train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

# Test
te_loss, te_acc = run_epoch(model, test_loader, train=False)
print("테스트 정확도:", te_acc)

device: cpu


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.03MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 138kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.07MB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


LeNet(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): Tanh()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): Tanh()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
    (7): Tanh()
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=120, out_features=84, bias=True)
    (2): Tanh()
    (3): Linear(in_features=84, out_features=10, bias=True)
  )
)


100%|██████████| 47/47 [00:01<00:00, 34.31it/s]


[1/10] train loss 0.4167 acc 0.8812 | val loss 0.1766 acc 0.9440


...

[9/10] train loss 0.0238 acc 0.9918 | val loss 0.0556 acc 0.9827


100%|██████████| 47/47 [00:01<00:00, 41.19it/s]


[10/10] train loss 0.0200 acc 0.9936 | val loss 0.0573 acc 0.9837


100%|██████████| 79/79 [00:01<00:00, 48.35it/s]

테스트 정확도: 0.9856


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

# -----------------------------
# 1) Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# -----------------------------
# 2) Dataset / Transform
# -----------------------------
BATCH_SIZE = 128
NUM_WORKERS = 4  # 환경에 따라 0~8 조절
IMG_SIZE = 227

# 전처리 파이프라인
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),    # (32, 32) -> (227, 227) 리사이즈
    transforms.ToTensor(),                      # [0,1] float32
])

test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_set = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tf)
test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# -----------------------------
# 3) AlexNet (CIFAR-10)
# -----------------------------
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # 합성곱 층 1
            nn.Conv2d(3, 96, kernel_size=11, stride=4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # 합성곱 층 2
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # 합성곱 층 3
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # 합성곱 층 4
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # 합성곱 층 5
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        # 227 입력이면 feature map이 6x6x256
        self.classifier = nn.Sequential(
            # 완전 연결 층 1
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            # 완전 연결 층 2
            nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            # 출력층
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = AlexNet(num_classes=10).to(device)
print("params:", sum(p.numel() for p in model.parameters())/1e6, "M")  # 파라미터 수 (MB)

# -----------------------------
# 4) Train / Eval
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in tqdm(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    te_loss, te_acc = evaluate(model, test_loader)
    print(f"[{epoch}/{EPOCHS}] train loss {tr_loss:.4f} acc {tr_acc:.4f} | test loss {te_loss:.4f} acc {te_acc:.4f}")

SyntaxError: invalid syntax (2029304129.py, line 102)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

# -----------------------------
# 1) Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# -----------------------------
# 2) Dataset / Transform
#    - CIFAR10 (32x32) -> 224x224 resize
# -----------------------------
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 0  # Windows/Jupyter 안정

mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_set = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tf)
test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=False)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=False)

# -----------------------------
# 3) VGG16-like model (원형에 가깝게)
#    - Block 구성 동일
#    - Classifier: Flatten -> 4096 -> 4096 -> 10
# -----------------------------
class VGG16Like(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # VGG 블록 (Conv 여러번 + Pool 1번) 생성
        def conv_block(in_c, out_c, n_conv):
            layers = []
            for i in range(n_conv):
                layers.append(nn.Conv2d(in_c if i == 0 else out_c, out_c, kernel_size=3, padding=1))  # 3x3 Conv
                layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))  # 다운샘플링 
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            conv_block(3,   64, 2),  # Block1
            conv_block(64, 128, 2),  # Block2
            conv_block(128,256, 3),  # Block3
            conv_block(256,512, 3),  # Block4
            conv_block(512,512, 3),  # Block5
        )

        # 224 -> pool 5번 -> 224/32 = 7
        # feature map: 512 x 7 x 7
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = VGG16Like(num_classes=10).to(device)
print("params:", sum(p.numel() for p in model.parameters())/1e6, "M")

# -----------------------------
# 4) Loss / Optimizer
#    - sparse_categorical_crossentropy == CrossEntropyLoss
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -----------------------------
# 5) Train / Eval loops
# -----------------------------
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

# -----------------------------
# 6) Train
# -----------------------------
EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    te_loss, te_acc = evaluate(model, test_loader)
    print(f"[{epoch}/{EPOCHS}] train loss {tr_loss:.4f} acc {tr_acc:.4f} | test loss {te_loss:.4f} acc {te_acc:.4f}")

print("테스트 정확도:", te_acc)

device: cpu


c:\Users\Playdata\multimodal\multi_venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


params: 134.301514 M


  6%|▌         | 24/391 [15:15<3:53:16, 38.14s/it]


KeyboardInterrupt: 